# 실습 04 · Transformer 구조·Shape (기본필수)

Token ID가 Transformer를 지나면서 **데이터 모양(shape)이 어떻게 바뀌는지** 따라갑니다.

`Token IDs (2, 5)` → `Embedding (2, 5, 8)` → `Encoder 출력 (2, 5, 8)`

| 기호 | 뜻 | 이 실습 값 |
|---|---|---|
| B | 한 번에 처리하는 문장 수 (batch) | 2 |
| T | 문장 길이 (Token 수) | 5 |
| D | Token 하나를 나타내는 숫자 개수 (d_model) | 8 |

### 사용 방법
1. **결과는 이미 셀 아래에 저장되어 있습니다.** 읽기만 해도 흐름을 따라갈 수 있어요.
2. 직접 돌려 보려면 `파일 → Drive에 사본 저장`으로 내 사본을 만드세요.
3. 맨 위 **⚙️ 준비 셀을 먼저 한 번 실행**한 뒤, 보고 싶은 구간의 셀을 실행하세요. 각 구간은 준비 셀만 있으면 따로 실행됩니다.
4. **✏️ 표시 셀**은 오른쪽 슬라이더의 값을 바꾸고 실행해 보세요. 코드의 값도 함께 바뀝니다.

> ⚠️ 가중치는 학습되지 않은 무작위 값입니다. 숫자 자체보다 **shape**에 집중하세요.

In [1]:
#@title ⚙️ 먼저 실행하세요 — 실습 공통 코드 준비
# ── 원본: 20261113/mentor/01_basic/common/support.py (그대로 포함) ──
"""강의 공통 오프라인 계산 도구: 실제 사전학습 LLM이 아닙니다.

교육용 정규표현식 토큰화·작은 벡터·가중치 계산만 제공합니다.
Transformer의 tokenizer/embedding 품질을 이 도구로 평가할 수 없습니다.
"""
from pathlib import Path
import json
import math
import re
import torch

DATA = [
    {"id":"A", "title":"실습 파일 보관", "content":"실습 파일은 개인 폴더에 보관합니다.", "category":"learning"},
    {"id":"B", "title":"서버 실행", "content":"프로젝트 폴더에서 서버를 실행합니다.", "category":"environment"},
    {"id":"C", "title":"질문 작성", "content":"오류 메시지와 시도한 작업을 함께 적습니다.", "category":"learning"},
]
SPECIAL = ["[PAD]", "[UNK]", "[BOS]", "[EOS]"]

def words(text: str) -> list[str]:
    """교육용: 한글 연속 글자/영문/숫자/문장부호 분리. 서브워드 토크나이저가 아님."""
    return re.findall(r'[가-힣]+|[A-Za-z]+|\d+|[^\w\s]', text)

class ToyTokenizer:
    """BERT tokenizer 대체가 아닌, 오프라인 개념 확인용 규칙 기반 어휘표."""
    def __init__(self, corpus):
        wordset=sorted({word for line in corpus for word in words(line)})
        self.vocab={tok:i for i,tok in enumerate(SPECIAL + wordset)}
        self.reverse={i:t for t,i in self.vocab.items()}

    def encode(self, text: str, max_length=None, padding_to=None):
        raw=words(text)
        tokens=["[BOS]"]+raw+["[EOS]"]
        truncated=False
        if max_length is not None:
            if max_length<2: raise ValueError("BOS/EOS를 포함하려면 max_length>=2")
            if len(tokens)>max_length:
                tokens=tokens[:max_length-1]+["[EOS]"]
                truncated=True
        valid=len(tokens)
        if padding_to is not None:
            if padding_to<valid: raise ValueError("padding_to must be >= valid length")
            tokens=tokens+["[PAD]"]*(padding_to-valid)
        ids=[self.vocab.get(t,self.vocab["[UNK]"]) for t in tokens]
        mask=[0 if t=="[PAD]" else 1 for t in tokens]
        return {"tokens":tokens,"input_ids":ids,"attention_mask":mask,
                "valid_length":valid,"truncated":truncated,"original_length":len(raw)+2}

    def batch(self, texts, max_length=None):
        items=[self.encode(t,max_length=max_length) for t in texts]
        max_size=max(len(i["input_ids"]) for i in items)
        return [self.encode(t,max_length=max_length,padding_to=max_size) for t in texts]

def context_budget(window, parts:dict, output_reserved:int):
    used=sum(parts.values())
    return {"window":window,"input_used":used,"output_reserved":output_reserved,
            "remaining":window-used-output_reserved,
            "fits":used+output_reserved<=window}

def cosine(a,b):
    a=torch.as_tensor(a,dtype=torch.float32)
    b=torch.as_tensor(b,dtype=torch.float32)
    return float(torch.nn.functional.cosine_similarity(a,b,dim=0))

def attention(q,k,v,causal=True):
    q=torch.as_tensor(q,dtype=torch.float32)
    k=torch.as_tensor(k,dtype=torch.float32)
    v=torch.as_tensor(v,dtype=torch.float32)
    scores=q@k.T/math.sqrt(q.shape[-1])
    mask=torch.triu(torch.ones_like(scores,dtype=torch.bool),diagonal=1)
    weights=torch.softmax(scores.masked_fill(mask,float('-inf')) if causal else scores,dim=-1)
    return scores,weights,weights@v,mask

def write_json(path, data):
    p=Path(path);p.parent.mkdir(exist_ok=True,parents=True)
    p.write_text(json.dumps(data,ensure_ascii=False,indent=2),encoding='utf-8')

def write_md(path,text):
    p=Path(path);p.parent.mkdir(exist_ok=True,parents=True)
    p.write_text(text,encoding='utf-8')

def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))

# ── 노트북 전용 준비 ──
import pandas as pd
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

OUT_DIR = Path("outputs")
OUT_DIR.mkdir(exist_ok=True)

from torch import nn

torch.manual_seed(42)
torch.set_num_threads(1)

# 실습 입력 (main.py와 같은 값): 문장 2개 × Token 5개, 0 = [PAD]
token_ids = torch.tensor([[2, 4, 5, 3, 0], [2, 6, 7, 3, 0]], dtype=torch.long)

def table(t, row_name="pos", digits=3):
    """2차원 텐서를 표로 보여 줌"""
    return pd.DataFrame(t.detach().numpy()).round(digits).rename_axis(row_name)

print("준비 완료 ✅  token_ids shape:", list(token_ids.shape))

준비 완료 ✅  token_ids shape: [2, 5]


## 0. 입력 Token ID
문장 2개, 각 5칸입니다. 맨 끝의 `0`은 길이를 맞추려고 채운 `[PAD]`입니다. (02 실습의 Padding)

In [2]:
pd.DataFrame(token_ids.numpy(), index=["문장1", "문장2"],
             columns=[f"t{i}" for i in range(token_ids.shape[1])])

,t0,t1,t2,t3,t4
문장1,2,4,5,3,0
문장2,2,6,7,3,0


In [3]:
print("token_ids shape (B, T):", list(token_ids.shape))

token_ids shape (B, T): [2, 5]


## B01. Embedding: ID → 벡터

`nn.Embedding`은 **ID 번호로 행을 찾아 오는 표**입니다. 16개 단어 × 8개 숫자 표에서 ID 행을 꺼내 옵니다.

### B01-1. Embedding 표 만들기

In [4]:
torch.manual_seed(42)                     # main.py와 같은 무작위 값
embedding = nn.Embedding(num_embeddings=16, embedding_dim=8)
print("Embedding 표 shape (단어 수, D):", list(embedding.weight.shape))

Embedding 표 shape (단어 수, D): [16, 8]


In [5]:
table(embedding.weight, "id").head(8)   # 표의 앞 8행

,0,1,2,3,4,5,6,7
id,,,,,,,,
0,1.927,1.487,0.901,-2.106,0.678,-1.235,-0.043,-1.605
1,-0.752,1.649,-0.392,-1.404,-0.728,-0.559,-0.769,0.762
2,1.642,-0.160,-0.497,0.440,-0.758,1.078,0.801,1.681
3,1.279,1.296,0.610,1.335,-0.232,0.042,-0.252,0.860
4,-1.385,-0.871,-0.223,1.717,0.319,-0.425,0.306,-0.775
5,-1.558,0.996,-0.880,-0.601,-1.274,2.123,-1.235,-0.488
6,-0.914,-0.658,0.078,0.526,-0.488,1.191,-0.814,-0.736
7,-1.403,0.036,-0.063,0.676,-0.098,1.845,-1.185,1.384


### B01-2. ID를 넣으면 벡터가 나온다

In [6]:
x = embedding(token_ids)
print("token_ids shape:", list(token_ids.shape), " → x shape:", list(x.shape))

token_ids shape: [2, 5]  → x shape: [2, 5, 8]


In [7]:
table(x[0], "문장1 위치")   # 문장1의 Token 5개 × 숫자 8개

,0,1,2,3,4,5,6,7
문장1 위치,,,,,,,,
0,1.642,-0.160,-0.497,0.440,-0.758,1.078,0.801,1.681
1,-1.385,-0.871,-0.223,1.717,0.319,-0.425,0.306,-0.775
2,-1.558,0.996,-0.880,-0.601,-1.274,2.123,-1.235,-0.488
3,1.279,1.296,0.610,1.335,-0.232,0.042,-0.252,0.860
4,1.927,1.487,0.901,-2.106,0.678,-1.235,-0.043,-1.605


### B01-3. 같은 ID는 같은 벡터
두 문장의 첫 Token은 모두 ID 2(`[BOS]`)입니다. 꺼내 온 벡터도 같고, Embedding 표의 2번 행과도 같습니다.

In [8]:
same = torch.equal(x[0, 0], x[1, 0]) and torch.equal(x[0, 0], embedding.weight[2])
print("문장1 첫 Token == 문장2 첫 Token == 표의 2번 행:", same)

문장1 첫 Token == 문장2 첫 Token == 표의 2번 행: True


In [9]:
assert list(token_ids.shape) == [2, 5] and list(x.shape) == [2, 5, 8] and same
print("B01 통과 ✅", list(token_ids.shape), "=>", list(x.shape))

B01 통과 ✅ [2, 5] => [2, 5, 8]


## B02. Encoder Layer: 모양은 그대로, 내용은 섞인다

Encoder Layer는 Attention(Token끼리 정보 주고받기)과 FFN(특징 변환)으로 이루어져 있습니다. **입력과 출력의 shape이 같아서** 여러 층을 쌓을 수 있습니다.

### B02-1. Layer 만들기

In [10]:
torch.manual_seed(42)
embedding = nn.Embedding(16, 8)
x = embedding(token_ids)

layer = nn.TransformerEncoderLayer(d_model=8, nhead=2, dim_feedforward=16,
                                   batch_first=True, dropout=0.0)
layer.eval()
print("d_model=8, nhead=2 → Head 하나가 맡는 차원:", 8 // 2)
print("학습할 파라미터 수:", sum(p.numel() for p in layer.parameters()))

d_model=8, nhead=2 → Head 하나가 맡는 차원: 4
학습할 파라미터 수: 600


### B02-2. Padding Mask
`True`인 칸(= PAD)은 Attention에서 **Key로 참고하지 않도록** 막습니다.

In [11]:
pad_mask = (token_ids == 0)
pd.DataFrame(pad_mask.numpy(), index=["문장1", "문장2"], columns=[f"t{i}" for i in range(5)])

,t0,t1,t2,t3,t4
문장1,False,False,False,False,True
문장2,False,False,False,False,True


### B02-3. Encoder 통과

In [12]:
with torch.no_grad():
    y = layer(x, src_key_padding_mask=pad_mask)
print("x shape:", list(x.shape), " → y shape:", list(y.shape))

x shape: [2, 5, 8]  → y shape: [2, 5, 8]


In [13]:
table(y[0], "문장1 위치")   # 같은 모양, 다른 값

,0,1,2,3,4,5,6,7
문장1 위치,,,,,,,,
0,1.295,-0.310,-1.441,-0.164,-1.222,0.199,0.034,1.609
1,-1.253,-0.906,0.046,2.175,0.500,-0.429,0.401,-0.534
2,-0.711,1.401,-0.590,-0.411,-0.685,1.938,-0.888,-0.054
3,1.491,0.894,-0.270,1.042,-1.102,-1.095,-1.168,0.209
4,1.578,0.611,0.257,-1.210,1.036,-1.318,-0.065,-0.890


### B02-4. 주의: PAD 위치의 출력은 0이 아니다
Padding Mask는 PAD를 **참고 대상(Key)** 에서 빼는 것이지, PAD 위치의 **출력**을 0으로 만들지 않습니다. 그래서 뒤 단계에서 PAD 위치 결과는 따로 무시해야 합니다.

In [14]:
print("문장1 PAD 위치(t4) 출력 절댓값 합:", round(y[0, 4].abs().sum().item(), 4))

문장1 PAD 위치(t4) 출력 절댓값 합: 6.9642


In [15]:
assert x.shape == y.shape
print("B02 통과 ✅", list(x.shape), "=>", list(y.shape))

B02 통과 ✅ [2, 5, 8] => [2, 5, 8]


### ✏️ B02 직접 해 보기 — 크기를 바꿔 보세요
D가 nhead로 나누어떨어지지 않으면 Layer를 만들 수 없습니다. 예: D=8, nhead=3.

In [16]:
#@title ✏️ B, T, D, nhead를 바꿔 보세요
B = 2  #@param {type:"slider", min:1, max:6, step:1}
T = 5  #@param {type:"slider", min:1, max:12, step:1}
D = 8  #@param {type:"slider", min:2, max:16, step:2}
nhead = 2  #@param {type:"slider", min:1, max:4, step:1}

ids = torch.randint(0, 16, (B, T))
try:
    my_layer = nn.TransformerEncoderLayer(d_model=D, nhead=nhead, dim_feedforward=16,
                                          batch_first=True, dropout=0.0).eval()
    with torch.no_grad():
        my_x = nn.Embedding(16, D)(ids)
        my_y = my_layer(my_x)
    print("Token IDs", list(ids.shape), "→ Embedding", list(my_x.shape), "→ Encoder", list(my_y.shape))
    print("✅ 입력과 출력 shape이 같습니다")
except AssertionError as e:
    print(f"❌ D={D}는 nhead={nhead}로 나누어떨어지지 않아 Layer를 만들 수 없습니다")

Token IDs [2, 5] → Embedding [2, 5, 8] → Encoder [2, 5, 8]
✅ 입력과 출력 shape이 같습니다


## 저장. 결과 저장 (`main.py`와 같은 결과)
모든 shape과 Mask를 `outputs/transformer_report.json`으로 저장합니다. 이 셀은 준비 셀만 있으면 혼자 실행됩니다.

In [17]:
torch.manual_seed(42)
embedding = nn.Embedding(num_embeddings=16, embedding_dim=8)
x = embedding(token_ids)
id_shape, emb_shape = list(token_ids.shape), list(x.shape)

layer = nn.TransformerEncoderLayer(d_model=8, nhead=2, dim_feedforward=16,
                                   batch_first=True, dropout=0.0)
layer.eval()
pad_mask = (token_ids == 0)
with torch.no_grad():
    y = layer(x, src_key_padding_mask=pad_mask)
assert x.shape == y.shape
out_shape = list(y.shape)

write_json(OUT_DIR / "transformer_report.json", {
    "ids": id_shape, "embedding": emb_shape, "encoder_output": out_shape,
    "pad_key_mask": pad_mask.tolist(), "padding_positions": [],
    "padding_output_abs_sum": None, "padding_note": "PAD Mask는 Causal Mask와 별개",
    "components": {"attention": "토큰 간 정보", "ffn": "특징 변환", "shape": out_shape},
    "bad_head_rejected": None, "extra_shapes": {},
    "architecture_note": "이 코드는 Encoder Layer의 Shape 실습",
})
print("[04]", id_shape, "=>", emb_shape, "=>", out_shape)

[04] [2, 5] => [2, 5, 8] => [2, 5, 8]


In [18]:
#@title 📥 결과 파일 내려받기 (Colab)
if IN_COLAB:
    from google.colab import files
    for name in ['transformer_report.json']:
        files.download(str(OUT_DIR / name))
else:
    print("Colab에서 실행하면 결과 파일이 내려받아집니다:", ['transformer_report.json'])

Colab에서 실행하면 결과 파일이 내려받아집니다: ['transformer_report.json']


---
### 정리
- **Embedding**: ID 하나 → 숫자 D개 벡터. `(B, T)` → `(B, T, D)`. 같은 ID는 같은 벡터.
- **Encoder Layer**: shape `(B, T, D)`를 유지한 채 Token끼리 정보를 섞는다. D는 nhead로 나누어떨어져야 한다.
- **Padding Mask**는 PAD를 참고 대상에서 뺄 뿐, PAD 위치 출력을 0으로 만들지 않는다.